# 11.4 高级RAG (Advanced RAG)

基础RAG在复杂场景下存在不足，高级RAG技术通过知识图谱、多模态检索和Agent增强来解决这些问题。

本节涵盖：
- GraphRAG与知识图谱检索
- RAG评估（RAGAS指标）
- 查询改写与分解
- 工业RAG系统架构

## 1. GraphRAG与知识图谱检索

**目的**：利用实体关系进行多跳推理，解决向量检索无法处理的关系推理问题

**GraphRAG核心流程**：
1. **实体抽取**：从文档中抽取实体和关系（LLM或NER模型）
2. **图谱构建**：构建实体-关系-实体的三元组图谱
3. **社区检测**：将图谱划分为社区，生成社区摘要
4. **检索**：先检索相关社区摘要，再在社区内检索具体实体
5. **生成**：基于图谱上下文生成答案

**vs 向量RAG**：
- 向量RAG：擅长语义相似度匹配，不擅长"A的经理的上级是谁"这类多跳查询
- GraphRAG：擅长关系推理和多跳查询，但构建成本高

**工业实践**：Microsoft GraphRAG、Neo4j + LLM

In [ ]:
from collections import defaultdict

class KnowledgeGraph:
    def __init__(self):
        self.entities = {}
        self.relations = []
        self.entity_relations = defaultdict(list)

    def add_entity(self, entity_id, entity_type, properties=None):
        self.entities[entity_id] = {'type': entity_type, 'properties': properties or {}}

    def add_relation(self, source, target, relation, properties=None):
        self.relations.append({'source': source, 'target': target, 'relation': relation, 'properties': properties or {}})
        self.entity_relations[source].append({'target': target, 'relation': relation})
        self.entity_relations[target].append({'target': source, 'relation': f'inverse_{relation}'})

    def get_neighbors(self, entity_id, relation_type=None):
        neighbors = self.entity_relations.get(entity_id, [])
        if relation_type:
            neighbors = [n for n in neighbors if n['relation'] == relation_type]
        return neighbors

    def multi_hop_search(self, start_entity, query_relations, max_hops=3):
        paths = [[start_entity]]
        all_paths = []
        for hop in range(max_hops):
            new_paths = []
            for path in paths:
                current = path[-1]
                neighbors = self.get_neighbors(current)
                for neighbor in neighbors:
                    if neighbor['target'] not in path:
                        new_path = path + [neighbor['target']]
                        new_paths.append(new_path)
                        if hop + 1 >= 2:
                            all_paths.append({'path': new_path, 'length': hop + 1})
            paths = new_paths
        return all_paths

kg = KnowledgeGraph()
kg.add_entity('alice', 'Person', {'role': 'Engineer'})
kg.add_entity('bob', 'Person', {'role': 'Manager'})
kg.add_entity('charlie', 'Person', {'role': 'Director'})
kg.add_entity('ml_team', 'Team', {'name': 'ML Team'})
kg.add_entity('eng_dept', 'Department', {'name': 'Engineering'})

kg.add_relation('alice', 'ml_team', 'member_of')
kg.add_relation('bob', 'ml_team', 'manages')
kg.add_relation('bob', 'eng_dept', 'member_of')
kg.add_relation('charlie', 'eng_dept', 'directs')

print('=== GraphRAG: Knowledge Graph Multi-hop Search ===')
print(f'Entities: {list(kg.entities.keys())}')
print(f'Relations: {[(r["source"], r["relation"], r["target"]) for r in kg.relations]}')

print(f'\nQuery: "Who is Alice\'s manager\'s director?"')
paths = kg.multi_hop_search('alice', max_hops=3)
for p in paths[:5]:
    path_str = ' → '.join(p['path'])
    print(f'  Path ({p["length"]} hops): {path_str}')

alice_neighbors = kg.get_neighbors('alice')
print(f'\nAlice\'s direct relations: {alice_neighbors}')
team = [n['target'] for n in alice_neighbors if n['relation'] == 'member_of']
if team:
    managers = kg.get_neighbors(team[0], 'manages')
    print(f'Team managers: {managers}')

print(f'\nKey: GraphRAG enables multi-hop reasoning through entity relationships.')
print(f'Vector RAG cannot answer "Who is Alice\'s manager\'s director?" effectively.')
print(f'Production: use Neo4j/NetworkX + LLM for entity extraction and graph construction.')

## 2. RAG评估（RAGAS指标）

**目的**：量化评估RAG系统的检索和生成质量

**RAGAS核心指标**：

| 指标 | 评估什么 | 计算方式 | 目标值 |
|------|---------|---------|--------|
| **Faithfulness** | 生成是否基于检索文档 | NLI验证每个claim | >0.85 |
| **Answer Relevance** | 回答是否切题 | 生成反向问题的相似度 | >0.80 |
| **Context Precision** | 检索文档中相关文档的排名 | 相关文档在top-k中的位置 | >0.75 |
| **Context Recall** | 是否检索到所有相关文档 | 答案信息被文档覆盖的比例 | >0.80 |

**评估流程**：
1. 准备测试集（question, answer, contexts, ground_truth）
2. 运行RAG系统生成回答
3. 用RAGAS计算各指标
4. 分析弱项，针对性优化

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class RAGASEvaluator:
    def __init__(self, d_model=64):
        self.nli_model = nn.Sequential(nn.Linear(d_model * 2, 32), nn.ReLU(), nn.Linear(32, 3))
        self.sim_model = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, d_model))

    def faithfulness(self, answer_embed, context_embeds):
        n_claims = answer_embed.shape[0]
        n_contexts = context_embeds.shape[0]
        supported = 0
        for i in range(n_claims):
            claim = answer_embed[i:i+1].expand(n_contexts, -1)
            combined = torch.cat([claim, context_embeds], dim=-1)
            nli_scores = self.nli_model(combined)
            if nli_scores[:, 0].max() > nli_scores[:, 1:].max():
                supported += 1
        return supported / max(n_claims, 1)

    def answer_relevance(self, question_embed, answer_embed):
        q_proj = self.sim_model(question_embed)
        a_proj = self.sim_model(answer_embed)
        sim = F.cosine_similarity(q_proj, a_proj, dim=-1)
        return sim.mean().item()

    def context_precision(self, relevance_labels):
        n = len(relevance_labels)
        cumsum = 0
        n_relevant = 0
        for i, label in enumerate(relevance_labels):
            if label:
                n_relevant += 1
                cumsum += n_relevant / (i + 1)
        return cumsum / max(n_relevant, 1)

    def context_recall(self, ground_truth_embeds, context_embeds):
        n_gt = ground_truth_embeds.shape[0]
        covered = 0
        for i in range(n_gt):
            gt = ground_truth_embeds[i:i+1].expand(context_embeds.shape[0], -1)
            sims = F.cosine_similarity(gt, context_embeds, dim=-1)
            if sims.max() > 0.7:
                covered += 1
        return covered / max(n_gt, 1)

evaluator = RAGASEvaluator()

question = torch.randn(1, 64)
answer = torch.randn(3, 64)
contexts = torch.randn(5, 64)
ground_truth = torch.randn(4, 64)
relevance_labels = [True, True, False, True, False]

print('=== RAGAS Evaluation ===')
faith = evaluator.faithfulness(answer, contexts)
rel = evaluator.answer_relevance(question, answer.mean(0, keepdim=True))
prec = evaluator.context_precision(relevance_labels)
recall = evaluator.context_recall(ground_truth, contexts)

print(f'Faithfulness:       {faith:.3f} (are claims supported by context?)')
print(f'Answer Relevance:   {rel:.3f} (does answer address the question?)')
print(f'Context Precision:  {prec:.3f} (are relevant docs ranked high?)')
print(f'Context Recall:     {recall:.3f} (are all relevant docs retrieved?)')

print(f'\n--- Optimization Guide ---')
if faith < 0.8:
    print(f'Low faithfulness → Improve prompt ("answer based ONLY on documents")')
if rel < 0.7:
    print(f'Low relevance → Improve query understanding / query rewriting')
if prec < 0.7:
    print(f'Low precision → Add reranking step with Cross-Encoder')
if recall < 0.7:
    print(f'Low recall → Increase top_k, add hybrid search, improve chunking')

print(f'\nKey: RAGAS provides a systematic framework for RAG evaluation.')
print(f'Each metric targets a specific component: retrieval, generation, or alignment.')
print(f'Production: use the ragas library for automated evaluation pipelines.')

## 3. 查询改写与分解

**目的**：优化用户查询，提升检索质量

**常见问题**：
- **模糊查询**："它是什么"→ 缺少上下文
- **复杂查询**："比较A和B的区别"→ 需要分别检索A和B
- **口语化查询**："怎么搞那个东西"→ 需要规范化

**改写策略**：
- **HyDE**：先让LLM生成假设性回答，用回答代替查询去检索
- **Query Decomposition**：将复杂查询分解为多个子查询
- **Step-back Prompting**：将具体问题抽象为更一般的问题
- **Multi-Query**：从不同角度生成多个查询，合并检索结果

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class QueryRewriter(nn.Module):
    def __init__(self, vocab_size=3000, d_model=64, d_embed=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 2, d_model*2, batch_first=True), 1
        )
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = self.encoder(h)
        return self.head(h)

class MultiQueryGenerator:
    def __init__(self, n_queries=3):
        self.n_queries = n_queries

    def generate_variants(self, query_embedding, n_variants=3):
        noise = torch.randn(n_variants, query_embedding.shape[-1]) * 0.1
        variants = query_embedding.unsqueeze(0) + noise
        return F.normalize(variants, dim=-1)

print('=== Query Rewriting Strategies ===')

original_query = "What is the difference between machine learning and deep learning?"
print(f'Original query: "{original_query}"')

print(f'\n--- Query Decomposition ---')
sub_queries = [
    "What is machine learning?",
    "What is deep learning?",
    "How does deep learning relate to machine learning?",
]
for i, sq in enumerate(sub_queries):
    print(f'  Sub-query {i+1}: "{sq}"')
print(f'Each sub-query is used for separate retrieval, results are merged.')

print(f'\n--- HyDE (Hypothetical Document Embedding) ---')
hyde_doc = "Machine learning is a broad field of AI. Deep learning is a specific subset that uses neural networks with multiple layers. The key difference is that deep learning automatically learns feature representations, while traditional ML often requires manual feature engineering."
print(f'HyDE document: "{hyde_doc[:80]}..."')
print(f'Use HyDE document embedding for retrieval instead of query embedding.')

print(f'\n--- Multi-Query ---')
mq_gen = MultiQueryGenerator(n_queries=3)
query_emb = F.normalize(torch.randn(32), dim=-1)
variants = mq_gen.generate_variants(query_emb, n_variants=3)
print(f'Generated {variants.shape[0]} query variants')
print(f'Each variant retrieves separately, results are union-ranked.')

print(f'\nKey: Query rewriting significantly improves retrieval quality.')
print(f'HyDE works well when queries are short/vague.')
print(f'Multi-query reduces missing relevant documents.')
print(f'Decomposition is essential for complex multi-aspect questions.')